# Modélisation Images — LeNet

L'objectif de ce notebook est de concevoir un premier modèle de classification d'images avec une architecture **LeNet**. C'est un CNN simple qui nous permet de mesurer la contribution des images seules à la prédiction, avant de passer au transfer learning (VGG16, notebook 6.0).

Nos ressources de calcul étant limitées sur Kaggle (GPU T4 x2, sessions de 12h), nous allons :

1. Tester l'impact de la **taille d'image** (128px vs 256px)
2. Tester l'impact du **volume de données** (25% vs dataset complet)
3. Conclure sur la pertinence de LeNet pour ce problème à 27 classes

| Modèle | Image | Data | Résultat |
|--------|-------|------|----------|
| 111    | 128px | 25%  | F1=0.2039 |
| 112    | 256px | 25%  | F1=0.1179 |
| 113    | 256px | 100% | Interrompu (val_f1=0.2065 ep.4) |

## Setup & Imports

In [ ]:
import os, sys, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, AveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import Callback, EarlyStopping
from tensorflow.keras import backend as K
import keras.saving
from keras.models import load_model

# ── Chemins Kaggle ───────────────────────────────────────────────────────────
INPUT_DIR  = Path('/kaggle/input/datasets/tarikouhsine/rakuten-preprocessed')
IMAGE_DIR  = INPUT_DIR / 'images' / 'images' / 'image_train'
WORK_DIR   = Path('/kaggle/working')
OUTPUT_DIR = WORK_DIR / 'modeling-lenet'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random_state = 42
start_lr     = 0.001

print(f'TensorFlow : {tf.__version__}')
print(f'GPU dispo  : {tf.config.list_physical_devices("GPU")}')
print(f'Image dir  : {IMAGE_DIR.exists()} ({len(list(IMAGE_DIR.glob("*.jpg"))):,} images)')

## Fonctions de reporting

Nous allons créer un dataframe `reporting_df` qui contiendra les informations pertinentes sur nos stratégies de modélisation ainsi que leurs résultats. Il sera sauvegardé dans `model_reporting.csv` pour assurer la résilience aux interruptions.

In [ ]:
output_dir = str(OUTPUT_DIR)

def create_reporting_df(save=True):
    reporting_df = pd.DataFrame([{
        'model_id'                    : int(0),
        'model_name'                  : 'Exemple',
        'dataset_resampled'           : False,
        'image_dataset_zoomed'        : False,
        'train_size'                  : int(0),
        'image_size'                  : int(128),
        'image_color_rgb'             : True,
        'batch_size'                  : int(32),
        'image_data_generator_params' : {},
        'start_lr'                    : 0.001,
        'last_lr'                     : 0.001,
        'optimizer'                   : 'adam',
        'target_epochs'               : int(10),
        'last_epoch'                  : int(0),
        'fit_complete'                : False,
        'f1_score'                    : 0.0,
        'val_f1_score'                : 0.0,
        'test_f1_score'               : 0.0,
        'accuracy'                    : 0.0,
        'val_accuracy'                : 0.0,
        'test_accuracy'               : 0.0,
        'loss'                        : 0.0,
        'val_loss'                    : 0.0,
        'remarks'                     : '',
    }])
    reporting_df.set_index('model_id', inplace=True)
    if save:
        reporting_df.to_csv(output_dir + '/model_reporting.csv')
    return reporting_df

def load_reporting_df():
    return pd.read_csv(output_dir + '/model_reporting.csv', index_col='model_id')

def add_reporting_line(model_id, model_name, save=True):
    reporting_df.loc[int(model_id), 'model_name'] = str(model_name)
    if save:
        reporting_df.to_csv(output_dir + '/model_reporting.csv')

def update_reporting_line(model_id, d={}, save=True):
    for key in d.keys():
        reporting_df.at[int(model_id), str(key)] = d.get(key)
    if save:
        reporting_df.to_csv(output_dir + '/model_reporting.csv')

def get_reporting_value(model_id, key, default=None):
    try:
        value = reporting_df.at[model_id, key]
        if pd.isna(value):
            value = None
    except KeyError:
        value = None
    if value is None and default is not None:
        value = default
    return value

print('✅ Fonctions reporting prêtes')

## Fonctions modèle, sauvegarde, évaluation

Note importante : `tf.py_function` et `tf.numpy_function` sont incompatibles avec la compilation XLA sur GPU Kaggle (T4). On implémente la métrique F1 en **pure TensorFlow** avec des accumulateurs.

In [ ]:
# ── Architecture LeNet ───────────────────────────────────────────────────────
def build_model(input_shape):
    """
    Architecture LeNet adaptée à 27 classes.
    CNN simple sans poids pré-entraînés — utilisé comme baseline.
    Params : ~1.6M (128px) | ~13.5M (256px)
    """
    model = Sequential([
        Conv2D(6,  (5, 5), activation='relu', input_shape=input_shape),
        AveragePooling2D(pool_size=(2, 2)),
        Conv2D(16, (5, 5), activation='relu'),
        AveragePooling2D(pool_size=(2, 2)),
        Flatten(),
        Dense(120, activation='relu'),
        Dense(84,  activation='relu'),
        Dense(27,  activation='softmax'),
    ])
    return model

# ── Métrique F1 — pure TensorFlow, compatible XLA GPU ────────────────────────
NUM_CLASSES = 27

@keras.saving.register_keras_serializable(name='WeightedF1Score')
class WeightedF1Score(tf.keras.metrics.Metric):
    """F1-score pondéré implémenté en pure TF — compatible XLA/GPU Kaggle"""

    def __init__(self, num_classes=NUM_CLASSES, name='w_f1_score', **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.tp      = self.add_weight(shape=(num_classes,), initializer='zeros', name='tp')
        self.fp      = self.add_weight(shape=(num_classes,), initializer='zeros', name='fp')
        self.fn      = self.add_weight(shape=(num_classes,), initializer='zeros', name='fn')
        self.support = self.add_weight(shape=(num_classes,), initializer='zeros', name='support')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true_cls = tf.argmax(y_true, axis=1)
        y_pred_cls = tf.argmax(y_pred, axis=1)
        y_true_oh  = tf.one_hot(y_true_cls, self.num_classes)
        y_pred_oh  = tf.one_hot(y_pred_cls, self.num_classes)
        self.tp.assign_add(tf.reduce_sum(y_true_oh * y_pred_oh,        axis=0))
        self.fp.assign_add(tf.reduce_sum((1 - y_true_oh) * y_pred_oh,  axis=0))
        self.fn.assign_add(tf.reduce_sum(y_true_oh * (1 - y_pred_oh),  axis=0))
        self.support.assign_add(tf.reduce_sum(y_true_oh,               axis=0))

    def result(self):
        eps       = tf.keras.backend.epsilon()
        precision = self.tp / (self.tp + self.fp + eps)
        recall    = self.tp / (self.tp + self.fn + eps)
        f1        = 2 * precision * recall / (precision + recall + eps)
        weights   = self.support / (tf.reduce_sum(self.support) + eps)
        return tf.reduce_sum(f1 * weights)

    def reset_state(self):
        zeros = tf.zeros(self.num_classes)
        self.tp.assign(zeros)
        self.fp.assign(zeros)
        self.fn.assign(zeros)
        self.support.assign(zeros)

    def get_config(self):
        config = super().get_config()
        config.update({'num_classes': self.num_classes})
        return config

# Instance globale réutilisée dans tous les modèles
w_f1_score = WeightedF1Score(num_classes=NUM_CLASSES)

# ── Sauvegarde / chargement modèle ───────────────────────────────────────────
def save_model_fn(model_id, model):
    path = output_dir + f'/model-{model_id}.keras'
    model.save(path)
    print(f'   💾 Modèle {model_id} sauvegardé → {path}')

def load_saved_model(model_id):
    path = Path(output_dir) / f'model-{model_id}.keras'
    if path.exists():
        print(f'   📂 Chargement modèle {model_id}')
        return load_model(path, custom_objects={'WeightedF1Score': WeightedF1Score})
    return None

# ── Historique d'entraînement ─────────────────────────────────────────────────
def save_history(model_id, history):
    with open(output_dir + f'/history_{model_id}.pkl', 'wb') as f:
        pickle.dump(history, f)

def load_history(model_id):
    path = Path(output_dir) / f'history_{model_id}.pkl'
    if path.exists():
        with open(path, 'rb') as f:
            return pickle.load(f)
    return None

def concat_histories(prev, curr):
    return {k: np.concatenate((prev[k], curr[k]), axis=0) for k in curr}

def get_full_history(model_id, training_history):
    if training_history is None:
        return load_history(model_id)
    prev = load_history(model_id)
    if prev is not None:
        return concat_histories(prev, training_history.history)
    return training_history.history

# ── Callback LogBest ──────────────────────────────────────────────────────────
class LogBest(Callback):
    """Sauvegarde le modèle et les métriques à chaque amélioration du val_f1."""

    def __init__(self, model_id):
        super().__init__()
        self.model_id = model_id

    def on_epoch_end(self, epoch, logs=None):
        update_reporting_line(self.model_id, {
            'last_epoch': epoch + 1,
            'last_lr'   : float(K.get_value(self.model.optimizer.learning_rate)),
        })
        if float(logs['val_w_f1_score']) <= float(
                get_reporting_value(self.model_id, 'val_f1_score', 0.0)):
            return
        print(f'\n   ✅ Meilleur modèle — val_f1={float(logs["val_w_f1_score"]):.4f}')
        update_reporting_line(self.model_id, {
            'f1_score'    : float(logs['w_f1_score']),
            'val_f1_score': float(logs['val_w_f1_score']),
            'accuracy'    : float(logs['accuracy']),
            'val_accuracy': float(logs['val_accuracy']),
            'loss'        : float(logs['loss']),
            'val_loss'    : float(logs['val_loss']),
        })
        save_model_fn(self.model_id, self.model)

# ── EarlyStopping ─────────────────────────────────────────────────────────────
early_stopping = EarlyStopping(
    monitor='val_w_f1_score', mode='max',
    patience=5, min_delta=1e-2
)

print('✅ Fonctions modèle prêtes — WeightedF1Score pure TF (compatible XLA GPU Kaggle)')

## Fonctions de visualisation

In [ ]:
CODE2LABEL = {
    10:'Livre occ.',    40:'Jeu vidéo',      50:'Access.Console',
    60:'Console',       1140:'Figurine',      1160:'Carte Coll.',
    1180:'Jeu Plateau', 1280:'Jouet enf.',    1281:'Jeu société',
    1300:'Jouet tech',  1301:'Chaussettes',   1302:'Jeu ext.',
    1320:'Bébé',        1560:'Mobilier int.', 1920:'Chambre',
    1940:'Cuisine',     2060:'Déco int.',     2220:'Animal',
    2280:'Revues',      2403:'Magazines',     2462:'Jeu occ.',
    2522:'Bureautique', 2582:'Mobilier ext.', 2583:'Piscine',
    2585:'Bricolage',   2705:'Livre neuf',    2905:'Jeu PC',
}

def plot_training_history(history):
    x = range(1, len(history['accuracy']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(x, history['val_w_f1_score'], label='val_f1')
    axes[0].plot(x, history['w_f1_score'],     label='train_f1')
    axes[0].plot(x, history['val_accuracy'],   label='val_accuracy')
    axes[0].plot(x, history['accuracy'],       label='train_accuracy')
    axes[0].set_title('F1 / Accuracy par époque')
    axes[0].set_xlabel('Époque')
    axes[0].legend()
    axes[1].plot(x, history['val_loss'], label='val_loss')
    axes[1].plot(x, history['loss'],     label='train_loss')
    axes[1].set_title('Loss par époque')
    axes[1].set_xlabel('Époque')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

def plot_confusion_matrix(cm):
    labels = [f'{c}: {CODE2LABEL.get(int(c), c)}' for c in classes_keras]
    plt.figure(figsize=(16, 10))
    sns.set(font_scale=0.7)
    sns.heatmap(cm, annot=True, fmt='g', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.xlabel('Prédit')
    plt.ylabel('Réel')
    plt.title('Matrice de confusion')
    plt.tight_layout()
    plt.show()

print('✅ Fonctions visualisation prêtes')

## Chargement des données

In [ ]:
df_train = pd.read_csv(INPUT_DIR / 'df_train.csv', index_col=0)
df_val   = pd.read_csv(INPUT_DIR / 'df_val.csv',   index_col=0)
df_test  = pd.read_csv(INPUT_DIR / 'df_test.csv',  index_col=0)

# Classes ordonnées — nécessaire pour flow_from_dataframe
classes_keras = list(np.sort(
    np.array(df_train['prdtypecode'].unique(), dtype='str')
))

print(f'Train : {df_train.shape} | Val : {df_val.shape} | Test : {df_test.shape}')
print(f'Classes : {len(classes_keras)}')
print(f'Colonnes : {df_train.columns.tolist()}')

# Construction colonne imagefile si absente
for df in [df_train, df_val, df_test]:
    if 'imagefile' not in df.columns:
        df['imagefile'] = ('image_' + df['imageid'].astype(str) +
                           '_product_' + df['productid'].astype(str) + '.jpg')

# Vérification images (structure plate — pas de sous-dossiers)
ok = sum(1 for f in df_train['imagefile'].head(20)
         if (IMAGE_DIR / f).exists())
print(f'\nVérification images (20 premières) : {ok}/20 trouvées')

## Initialisation du rapport de modèle

In [ ]:
try:
    reporting_df = load_reporting_df()
    print('✅ Reporting chargé')
except FileNotFoundError:
    reporting_df = create_reporting_df()
    print('✅ Nouveau reporting créé')

display(reporting_df.head())

## Générateurs d'images (configuration partagée)

In [ ]:
batch_size      = 64
target_img_size = (128, 128, 3)  # modifié selon les modèles
color_mode      = 'rgb'

train_data_generator = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=5,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)
val_test_generator = ImageDataGenerator(rescale=1./255)

def make_generators(df_tr, df_v, df_te, img_size, bs):
    """Crée les 3 générateurs train/val/test.
    Les images sont dans une structure PLATE (pas de sous-dossiers par classe).
    """
    def flow(df, gen, shuffle):
        return gen.flow_from_dataframe(
            dataframe=pd.DataFrame({
                'imagefile'  : df['imagefile'],
                'prdtypecode': df['prdtypecode'].astype('str')
            }),
            x_col='imagefile',
            y_col='prdtypecode',
            directory=str(IMAGE_DIR),
            target_size=img_size[:2],
            color_mode=color_mode,
            batch_size=bs,
            class_mode='categorical',
            classes=classes_keras,
            shuffle=shuffle
        )
    return (flow(df_tr, train_data_generator, True),
            flow(df_v,  val_test_generator,   False),
            flow(df_te, val_test_generator,   False))

print('✅ Générateurs configurés')

---
## Modélisation 1.1 — LeNet 128px, dataset 25%

Premier test : architecture LeNet, images 128×128, sous-ensemble stratifié de 25% du dataset rééchantillonné (27 391 images).
Objectif : obtenir un score de référence rapidement (~5 min/epoch).

In [ ]:
model_id        = 111
target_epochs   = 10
target_img_size = (128, 128, 3)

add_reporting_line(model_id, 'LeNet 128px - 25% dataset')
update_reporting_line(model_id, {
    'dataset_resampled'           : True,
    'image_dataset_zoomed'        : True,
    'image_size'                  : 128,
    'batch_size'                  : batch_size,
    'image_color_rgb'             : True,
    'start_lr'                    : start_lr,
    'optimizer'                   : 'adam',
    'target_epochs'               : target_epochs,
    'image_data_generator_params' : {
        'rescale': 1./255, 'rotation_range': 20,
        'width_shift_range': 0.2, 'height_shift_range': 0.2,
        'shear_range': 5, 'zoom_range': 0.1, 'horizontal_flip': True
    }
})

# Sous-ensemble 25% stratifié
target_size = int(len(df_train) * 0.25)
df_sub, _   = train_test_split(df_train, train_size=target_size,
                               random_state=random_state,
                               stratify=df_train['prdtypecode'])

update_reporting_line(model_id, {'train_size': len(df_sub)})

train_gen, val_gen, test_gen = make_generators(
    df_sub, df_val, df_test, target_img_size, batch_size)

model = load_saved_model(model_id)
if model is None:
    model = build_model(target_img_size)
    model.compile(optimizer=Adam(learning_rate=start_lr),
                  loss='categorical_crossentropy',
                  metrics=['accuracy', w_f1_score])

model.summary()

In [ ]:
if not get_reporting_value(model_id, 'fit_complete', False):
    training_history = model.fit(
        train_gen,
        validation_data=val_gen,
        steps_per_epoch=train_gen.samples // batch_size,
        validation_steps=val_gen.samples // batch_size,
        epochs=target_epochs - int(get_reporting_value(model_id, 'last_epoch', 0)),
        callbacks=[LogBest(model_id), early_stopping]
    )
    update_reporting_line(model_id, {'fit_complete': True})
else:
    print('Déjà entraîné')
    training_history = None

In [ ]:
# Évaluation sur le jeu de test
model = load_saved_model(model_id)

test_pred       = model.predict(test_gen, verbose=1)
test_pred_class = [classes_keras[c] for c in np.argmax(test_pred, axis=1)]

f1_test  = f1_score(df_test['prdtypecode'].astype('str'), test_pred_class, average='weighted')
acc_test = accuracy_score(df_test['prdtypecode'].astype('str'), test_pred_class)

update_reporting_line(model_id, {'test_f1_score': f1_test, 'test_accuracy': acc_test})

print(f'F1 Test  : {f1_test:.4f}')
print(f'Accuracy : {acc_test:.4f}')

In [ ]:
history = get_full_history(model_id, training_history)
if history is not None:
    save_history(model_id, history)
    training_history = None
    plot_training_history(history)

print(classification_report(df_test['prdtypecode'].astype('str'), test_pred_class))

cm = pd.crosstab(df_test['prdtypecode'], test_pred_class,
                 rownames=['réel'], colnames=['prédit'])
plot_confusion_matrix(np.array(cm))

In [ ]:
# Résumé modèle 111
reporting_df.loc[model_id].T

---
## Modélisation 1.2 — LeNet 256px, dataset 25%

Même configuration que 1.1 mais avec des images **256×256**.
Hypothèse : une résolution plus haute fournirait plus de détails visuels.

Note : les étapes par epoch passent de 427 à 213 (même dataset, batch_size identique) car `steps_per_epoch = samples // batch_size`.

In [ ]:
model_id        = 112
target_epochs   = 10
target_img_size = (256, 256, 3)

add_reporting_line(model_id, 'LeNet 256px - 25% dataset')
update_reporting_line(model_id, {
    'dataset_resampled'           : True,
    'image_dataset_zoomed'        : True,
    'image_size'                  : 256,
    'batch_size'                  : batch_size,
    'image_color_rgb'             : True,
    'start_lr'                    : start_lr,
    'optimizer'                   : 'adam',
    'target_epochs'               : target_epochs,
    'image_data_generator_params' : {
        'rescale': 1./255, 'rotation_range': 20,
        'width_shift_range': 0.2, 'height_shift_range': 0.2,
        'shear_range': 5, 'zoom_range': 0.1, 'horizontal_flip': True
    }
})

# Même sous-ensemble 25%
target_size = int(len(df_train) * 0.25)
df_sub, _   = train_test_split(df_train, train_size=target_size,
                               random_state=random_state,
                               stratify=df_train['prdtypecode'])

update_reporting_line(model_id, {'train_size': len(df_sub)})

train_gen, val_gen, test_gen = make_generators(
    df_sub, df_val, df_test, target_img_size, batch_size)

model = load_saved_model(model_id)
if model is None:
    model = build_model(target_img_size)
    model.compile(optimizer=Adam(learning_rate=start_lr),
                  loss='categorical_crossentropy',
                  metrics=['accuracy', w_f1_score])

model.summary()

In [ ]:
if not get_reporting_value(model_id, 'fit_complete', False):
    training_history = model.fit(
        train_gen,
        validation_data=val_gen,
        steps_per_epoch=train_gen.samples // batch_size,
        validation_steps=val_gen.samples // batch_size,
        epochs=target_epochs - int(get_reporting_value(model_id, 'last_epoch', 0)),
        callbacks=[LogBest(model_id), early_stopping]
    )
    update_reporting_line(model_id, {'fit_complete': True})
else:
    print('Déjà entraîné')
    training_history = None

In [ ]:
model = load_saved_model(model_id)

test_pred       = model.predict(test_gen, verbose=1)
test_pred_class = [classes_keras[c] for c in np.argmax(test_pred, axis=1)]

f1_test  = f1_score(df_test['prdtypecode'].astype('str'), test_pred_class, average='weighted')
acc_test = accuracy_score(df_test['prdtypecode'].astype('str'), test_pred_class)

update_reporting_line(model_id, {'test_f1_score': f1_test, 'test_accuracy': acc_test})

print(f'F1 Test  : {f1_test:.4f}')
print(f'Accuracy : {acc_test:.4f}')

In [ ]:
history = get_full_history(model_id, training_history)
if history is not None:
    save_history(model_id, history)
    training_history = None
    plot_training_history(history)

print(classification_report(df_test['prdtypecode'].astype('str'), test_pred_class))

cm = pd.crosstab(df_test['prdtypecode'], test_pred_class,
                 rownames=['réel'], colnames=['prédit'])
plot_confusion_matrix(np.array(cm))

In [ ]:
reporting_df.loc[model_id].T

---
## Modélisation 1.3 — LeNet 256px, dataset complet (interrompu)

Test avec le dataset complet (109 566 images) et images 256px avec lr_decay.

**Résultat** : session interrompue après 4 epochs (val_f1=0.2065).
Cause : ~28 min/epoch × 20 epochs = ~9h → dépasse la session Kaggle.

Les résultats partiels sont conservés pour comparaison.

In [ ]:
# Enregistrement des résultats partiels de la modélisation 1.3
model_id = 113

add_reporting_line(model_id, 'LeNet 256px - dataset complet (interrompu ep.4/20)')
update_reporting_line(model_id, {
    'dataset_resampled'           : True,
    'image_dataset_zoomed'        : True,
    'train_size'                  : len(df_train),   # 109 566
    'image_size'                  : 256,
    'batch_size'                  : 64,
    'image_color_rgb'             : True,
    'start_lr'                    : 0.001,
    'optimizer'                   : 'adam + CosineAnnealingLR',
    'target_epochs'               : 20,
    'last_epoch'                  : 4,
    'fit_complete'                : False,
    'val_f1_score'                : 0.2065,
    'val_accuracy'                : 0.2124,
    'val_loss'                    : 2.7565,
    'remarks'                     : 'Interrompu ep.4/20 — 28min/epoch sur T4'
})

print('Résultats partiels modèle 113 enregistrés.')
print(f'  val_f1 (ep.4) = 0.2065')
print(f'  Comparaison : 111 (128px) → 0.1992 | 112 (256px) → 0.1219')
print(f'  Le dataset complet aide légèrement mais le temps est prohibitif.')

---
## Comparaison des modélisations LeNet

In [ ]:
cols = ['model_name', 'train_size', 'image_size', 'batch_size',
        'target_epochs', 'last_epoch', 'fit_complete', 'val_f1_score', 'test_f1_score']
reporting_df = load_reporting_df()

ids_to_show = [id_ for id_ in [111, 112, 113] if id_ in reporting_df.index]
display(reporting_df.loc[ids_to_show, cols].T)

In [ ]:
# Barplot de comparaison — tous modèles du projet
modeles = ['LeNet\n128px', 'LeNet\n256px', 'VGG16\n(Transfer)', 'TF-IDF\n+SVC', 'CamemBERT']
scores  = [0.2039, 0.1179, 0.6521, 0.8121, 0.8826]
colors  = ['#e74c3c', '#e74c3c', '#f39c12', '#3498db', '#2ecc71']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(modeles, scores, color=colors, edgecolor='white', linewidth=1.5, width=0.6)
ax.set_ylim(0, 1.0)
ax.set_ylabel('F1-score weighted (test)', fontsize=12)
ax.set_title('Comparaison des modèles — Rakuten Classification', fontsize=14, fontweight='bold')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Seuil 0.5')

for bar, val in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontweight='bold', fontsize=10)

ax.legend()
plt.tight_layout()
plt.savefig(WORK_DIR / 'comparaison_tous_modeles.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Conclusion

### Résultats LeNet

| Modèle | Image | Train | Epochs | Val F1 | Test F1 |
|--------|-------|-------|--------|--------|---------|
| 111 LeNet 128px | 128px | 25% (27k) | 10 | 0.1992 | **0.2039** |
| 112 LeNet 256px | 256px | 25% (27k) | 10 | 0.1219 | **0.1179** |
| 113 LeNet 256px | 256px | 100% (109k) | 4/20 | 0.2065 | — |

### Observations

**1. 128px > 256px avec LeNet** : la résolution plus haute ne compense pas la faible capacité du réseau. LeNet (~1.6M paramètres en 128px) n'a pas la profondeur nécessaire pour extraire des features complexes depuis des images de haute résolution.

**2. Le dataset complet aide légèrement** : val_f1=0.2065 au bout de 4 epochs vs 0.1992 en 10 epochs avec 25% des données. Mais le temps de calcul est prohibitif (~28min/epoch sur T4), rendant l'entraînement complet irréalisable dans une session Kaggle (max 12h).

**3. LeNet est inadapté pour 27 classes visuellement complexes** : F1~0.20 confirme qu'un CNN simple sans poids pré-entraînés ne peut pas apprendre des représentations visuelles suffisamment riches pour distinguer 27 catégories de produits.

### Conclusion principale

Ces résultats justifient pleinement le passage au **transfer learning** (notebook 6.0 — VGG16) :
- VGG16 → F1=**0.6521** (×3.2 par rapport au meilleur LeNet)
- CamemBERT → F1=**0.8826** (le texte reste la modalité la plus discriminante)

L'image seule avec un CNN simple est insuffisante. La fusion texte+image (notebook 7.0) est la piste la plus prometteuse.